# Ethiopia Climate EDA\n
This notebook implements Task 2 for Ethiopia: profiling, cleaning, and exploratory analysis.

In [ ]:
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
from scipy.stats import zscore

In [ ]:
country = 'Ethiopia'\n
df = pd.read_csv('../data/ethiopia.csv')\n
df['Country'] = country\n
df['DATE'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')\n
df['Month'] = df['DATE'].dt.month\n
df = df.replace(-999, np.nan)\n
duplicate_count = df.duplicated().sum()\n
df = df.drop_duplicates()\n
print('Duplicates found:', duplicate_count)\n
df.head()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns\n
display(df[numeric_cols].describe())\n
missing = df.isna().sum().to_frame('missing_count')\n
missing['missing_pct'] = missing['missing_count'] / len(df) * 100\n
display(missing.sort_values('missing_pct', ascending=False))

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']\n
z_df = df[z_cols].copy()\n
z_df = z_df.fillna(z_df.median(numeric_only=True))\n
z_values = np.abs(zscore(z_df, nan_policy='omit'))\n
outlier_mask = (z_values > 3).any(axis=1)\n
print('Outlier rows (|Z| > 3):', int(outlier_mask.sum()))

In [ ]:
row_missing_ratio = df.isna().mean(axis=1)\n
df = df[row_missing_ratio <= 0.30].copy()\n
weather_cols = ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']\n
for col in weather_cols:\n
    if col in df.columns:\n
        df[col] = df[col].ffill()\n
df.to_csv('../data/ethiopia_clean.csv', index=False)\n
df.shape

In [ ]:
monthly_temp = df.resample('M', on='DATE')['T2M'].mean()\n
warmest = monthly_temp.idxmax()\n
coolest = monthly_temp.idxmin()\n
plt.figure(figsize=(12, 4))\n
monthly_temp.plot()\n
plt.scatter([warmest, coolest], [monthly_temp.max(), monthly_temp.min()], color=['red', 'blue'])\n
plt.title('Monthly Average T2M (Ethiopia)')\n
plt.ylabel('Temperature (°C)')\n
plt.show()

In [ ]:
monthly_prec = df.resample('M', on='DATE')['PRECTOTCORR'].sum()\n
plt.figure(figsize=(12, 4))\n
monthly_prec.plot(kind='bar')\n
plt.title('Monthly Total PRECTOTCORR (Ethiopia)')\n
plt.ylabel('Precipitation (mm/month)')\n
plt.tight_layout()\n
plt.show()

In [ ]:
corr = df.select_dtypes(include='number').corr(numeric_only=True)\n
plt.figure(figsize=(10, 8))\n
sns.heatmap(corr, cmap='coolwarm', center=0)\n
plt.title('Correlation Heatmap')\n
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=axes[0])\n
axes[0].set_title('T2M vs RH2M')\n
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=axes[1])\n
axes[1].set_title('T2M_RANGE vs WS2M')\n
plt.tight_layout()\n
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))\n
sns.histplot(df['PRECTOTCORR'], bins=50)\n
plt.yscale('log')\n
plt.title('PRECTOTCORR Distribution (log-y scale)')\n
plt.show()\n
\n
plt.figure(figsize=(10, 6))\n
sns.scatterplot(data=df, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(10, 200), alpha=0.5)\n
plt.title('Bubble Chart: T2M vs RH2M (size = PRECTOTCORR)')\n
plt.show()